In [1]:
# Jalankan sekali jika pustaka belum tersedia:
# !pip install yfinance pandas

from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import yfinance as yf

# Pengaturan sumber data
TICKER = "^JKSE"                 # Simbol IHSG di Yahoo Finance
TANGGAL_AWAL = "2010-01-01"
TANGGAL_AKHIR = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")

# Folder keluaran
folder_data = Path("data")
folder_data.mkdir(exist_ok=True)

# Mengunduh OHLCV harian tanpa penyesuaian otomatis
data = yf.download(
    tickers=TICKER,
    start=TANGGAL_AWAL,
    end=TANGGAL_AKHIR,
    interval="1d",
    auto_adjust=False,
    progress=False,
    group_by="column",
    threads=False
)

if data.empty:
    raise ValueError("Pengunduhan gagal atau data IHSG tidak ditemukan.")

# Merapikan nama kolom jika yfinance menghasilkan MultiIndex
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# Menjadikan tanggal sebagai kolom biasa
data = data.reset_index()
data.columns = [str(kolom).strip() for kolom in data.columns]

# Menambahkan metadata agar sumber dapat diaudit
data["Ticker"] = TICKER
data["Data_Source"] = "Yahoo Finance"
data["Download_Time"] = datetime.now().isoformat(timespec="seconds")

# Menyimpan data mentah; jangan ditimpa saat proses pembersihan
waktu_unduh = datetime.now().strftime("%Y%m%d_%H%M%S")
lokasi_raw = folder_data / f"ihsg_raw_{waktu_unduh}.csv"
data.to_csv(lokasi_raw, index=False, encoding="utf-8-sig")

print(f"Data tersimpan: {lokasi_raw}")
print(f"Jumlah observasi: {len(data):,}")
print(f"Periode: {data['Date'].min()} sampai {data['Date'].max()}")
display(data.head())

Data tersimpan: data/ihsg_raw_20260729_005741.csv
Jumlah observasi: 4,017
Periode: 2010-01-04 00:00:00 sampai 2026-07-28 00:00:00


,Date,Adj Close,Close,High,Low,Open,Volume,Ticker,Data_Source,Download_Time
0,2010-01-04,2575.312988,2575.413086,2576.055908,2532.895996,2533.947998,18339300,^JKSE,Yahoo Finance,2026-07-29T00:57:41
1,2010-01-05,2605.175537,2605.277100,2606.069092,2575.616943,2575.616943,57043800,^JKSE,Yahoo Finance,2026-07-29T00:57:41
2,2010-01-06,2603.195557,2603.297119,2622.115967,2587.709961,2605.480957,51569100,^JKSE,Yahoo Finance,2026-07-29T00:57:41
3,2010-01-07,2586.794189,2586.895020,2611.603027,2570.272949,2603.500977,45510800,^JKSE,Yahoo Finance,2026-07-29T00:57:41
4,2010-01-08,2614.268311,2614.370117,2614.535889,2583.846924,2586.792969,73723500,^JKSE,Yahoo Finance,2026-07-29T00:57:41
